# CS 3892 / 5892 — HW5 · Neural-network verification

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ttj/cs3892-examples/blob/main/notebooks/hw5-nn-verification.ipynb)

**Out Thu Nov 19 · due Fri Dec 4, 11:59 p.m.** The handout is
[`assignments/hw5.md`](https://github.com/ttj/cs3892-fmstai-fall2026/blob/main/assignments/hw5.md) in the class
repository. This notebook installs the pinned tools and runs the starters as they ship: every part is
still a placeholder. Nothing here is an answer.

The network classifies 14×14 MNIST digits (196 → 32 → 32 → 10, ReLU). Three tools answer each query:
**CROWN** (auto_LiRPA; sound, incomplete), an exact **MILP** solved by HiGHS (sound and complete), and a
**PGD** attack (finds counterexamples, proves nothing).

**To work here:** open the folder icon (left) → `cs3892-examples/homework/hw5-nn-verification/`, double-click a
`part….py` file or a `part5/….vnnlib` file, edit, save (Ctrl+S), and re-run the cells below.
**A Colab runtime is deleted when it disconnects: download your files before you leave.**

## Setup

Run these two once.

In [ ]:
# --- Setup: find the repo (clone on Colab), define helpers
import os, subprocess, sys, pathlib

REPO_URL = "https://github.com/ttj/cs3892-examples.git"
HW = "hw5-nn-verification"

def _find_repo():
    here = pathlib.Path.cwd()
    for p in [here, *here.parents]:
        if (p / "homework" / HW).is_dir():
            return p
    dest = pathlib.Path("/content/cs3892-examples") if pathlib.Path("/content").is_dir() \
           else pathlib.Path.cwd() / "cs3892-examples"
    if not (dest / "homework" / HW).is_dir():
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(dest)], check=True)
    return dest

ROOT = _find_repo()
HD = ROOT / "homework" / HW
PYD = HD / "python"
print("homework folder:", HD)

def show(path, lines=None):
    path = pathlib.Path(path)
    text = path.read_text().rstrip().splitlines()
    print(f"--- {path.name} " + "-" * max(0, 60 - len(path.name)))
    print("\n".join(text if lines is None else text[:lines]))
    print()

def run(path, *args, ok=(0,)):
    """Run a script from the homework folder and print its output (exit codes in `ok` are fine)."""
    path = pathlib.Path(path)
    r = subprocess.run([sys.executable, str(path), *map(str, args)], capture_output=True, text=True,
                       cwd=str(HD))
    print(r.stdout.rstrip())
    if r.returncode not in ok:
        print(r.stderr.rstrip()[-3000:], file=sys.stderr)
        raise RuntimeError(f"{path.name} exited with {r.returncode}")
    return r.stdout

def check(strict=False):
    """Report each part: ok / placeholder / wrong / error (and whether the tools are the pinned ones)."""
    return run(HD / "check_hw5.py", *(["--strict"] if strict else []), ok=(0, 1))

print("ready -- helpers: show(path), run(path), check(strict=False)")

In [ ]:
# --- The tools, every version pinned (requirements.txt, requirements-lirpa.txt): torch 2.11.0 CPU
# (about 200 MB; it replaces Colab's own torch, which is too new for auto_LiRPA 0.8.0), auto_LiRPA
# 0.8.0 at its release commit, HiGHS (highspy) and numpy. About a minute on Colab.
pip = [sys.executable, "-m", "pip", "install", "--quiet"]
subprocess.run(pip + ["-r", str(HD / "requirements.txt")], check=True)
subprocess.run(pip + ["--no-deps", "--ignore-requires-python", "-r", str(HD / "requirements-lirpa.txt")], check=True)
out = check()
assert "tools as pinned" in out, "the installed tools are not the pinned versions"

## The three tools

`hw5lib.py` holds the network, the images, the VNN-LIB parser and meaning check, and the three tools.
The checker pins it by SHA-256, so read it but leave it as it is.

In [ ]:
show(HD / "hw5lib.py", lines=40)

## Parts 1–4: the starters

Session 24 (Thu Nov 19) covers Parts 1, 2 and 4; session 25 (Tue Dec 1) covers Parts 3 and 5. Part 1's
`make_vnnlib` feeds Parts 2, 3 and 4, so do it first.

In [ ]:
for f in ("part1_query.py", "part2_frontier.py", "part3_compare.py", "part4_bounds.py"):
    show(HD / f)

In [ ]:
# Each part, as it ships: Part 1's make_vnnlib is still the placeholder, so they stop there.
for f in ("part1_query.py", "part2_frontier.py", "part3_compare.py", "part4_bounds.py"):
    print(f"=== {f}")
    run(HD / f, ok=(0, 1))

## Part 5: properties in English, VNN-LIB from an LLM

In [ ]:
show(HD / "part5" / "properties.md")

## What the checker and the tools are tested against

`python/00_hw5_starter_checks.py` is the course's own test. The meaning check must catch four typical
VNN-LIB mistakes (the safe condition instead of the counterexample, `and` for `or`, no clipping to
[0, 1], the wrong eps). CROWN must never verify a query the MILP falsifies; every counterexample must
survive a forward pass; and CROWN's "unknown" must be decided by the MILP at least once.

In [ ]:
out = run(PYD / "00_hw5_starter_checks.py")
assert "soundness: 12 queries consistent" in out and "exit 2" in out

## Your progress

Re-run this after every edit. `check(strict=True)` is the acceptance test: it fails unless every part
is OK. The written answers and Part 4 are graded by hand.

In [ ]:
out = check()
assert "Part 1:" in out and "Part 5:" in out